# Customer Service Conditional Routing with LangGraph

This notebook demonstrates **conditional routing** in LangGraph for a customer support system.

The workflow:
1. Receives a customer support request with a message and priority level
2. Categorizes the request based on priority
3. Routes to either **Urgent Support Team** (priority 1-2) or **Standard Support Queue** (priority 3+)

This is a common pattern in real-world applications where different paths are taken based on input conditions.

## Cell 1: Load Environment Variables

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables (override=True to use .env file over system env)
load_dotenv(override=True)

# Verify API key is loaded (optional for this demo, but good practice)
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f"✓ OpenAI API Key loaded: {api_key[:15]}...{api_key[-5:]}")
else:
    print("⚠ OpenAI API Key NOT found (not required for this basic demo)")

## Cell 2: Import Required Libraries

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import END, START, StateGraph

print("✓ All libraries imported successfully!")

## Cell 3: Define the State Schema

The state defines the structure of data flowing through our graph.

In [ ]:
# Define the structure of the input state (customer support request)
class SupportRequest(TypedDict):
    message: str
    priority: int  # 1 (high), 2 (medium), 3 (low)
    status: str    # Track the routing status

print("✓ SupportRequest state schema defined")
print("  Fields:")
print("    - message: str (customer's message)")
print("    - priority: int (1=high, 2=medium, 3=low)")
print("    - status: str (routing status)")

## Cell 4: Define the Categorization Node

This node receives the request and prepares it for routing.

In [ ]:
# Function to categorize the support request
def categorize_request(request: SupportRequest):
    """Initial node that receives and logs the request"""
    print(f"📥 Received request: '{request['message']}'")
    print(f"   Priority Level: {request['priority']}")
    return {"status": "categorized"}

print("✓ categorize_request node defined")

## Cell 5: Define the Routing Function (Conditional Edge)

This function determines which path to take based on the priority level.

In [ ]:
# Conditional routing function
def route_by_priority(request: SupportRequest) -> Literal["handle_urgent", "handle_standard"]:
    """
    Route based on priority:
    - Priority 1 or 2 → Urgent Support Team
    - Priority 3 or higher → Standard Support Queue
    """
    if request["priority"] <= 2:
        print(f"🚨 HIGH PRIORITY detected → Routing to URGENT team")
        return "handle_urgent"
    else:
        print(f"📋 Standard priority → Routing to STANDARD queue")
        return "handle_standard"

print("✓ route_by_priority conditional function defined")
print("  Routing logic:")
print("    - Priority 1-2 → handle_urgent")
print("    - Priority 3+  → handle_standard")

## Cell 6: Define Handler Nodes

These nodes process requests based on their routing.

In [ ]:
# Function to process high-priority requests
def handle_urgent(request: SupportRequest):
    """Handle urgent/high-priority support requests"""
    print(f"🔴 URGENT SUPPORT TEAM processing: '{request['message']}'")
    print(f"   → Escalating to senior agent...")
    print(f"   → Sending immediate notification...")
    return {"status": "urgent_handled"}


# Function to process standard requests
def handle_standard(request: SupportRequest):
    """Handle standard/low-priority support requests"""
    print(f"🟢 STANDARD SUPPORT QUEUE processing: '{request['message']}'")
    print(f"   → Adding to ticket queue...")
    print(f"   → Estimated response time: 24-48 hours")
    return {"status": "standard_handled"}

print("✓ Handler nodes defined:")
print("  - handle_urgent: For priority 1-2 requests")
print("  - handle_standard: For priority 3+ requests")

## Cell 7: Build the LangGraph Workflow

Now we connect all the nodes with edges, including the conditional routing.

In [ ]:
# Create the state graph
graph = StateGraph(SupportRequest)

# Add nodes
graph.add_node("categorize_request", categorize_request)
graph.add_node("handle_urgent", handle_urgent)
graph.add_node("handle_standard", handle_standard)

# Add edges
# START → categorize_request
graph.add_edge(START, "categorize_request")

# Conditional edge: categorize_request → (handle_urgent OR handle_standard)
graph.add_conditional_edges(
    "categorize_request",
    route_by_priority,
    {
        "handle_urgent": "handle_urgent",
        "handle_standard": "handle_standard"
    }
)

# Both handlers → END
graph.add_edge("handle_urgent", END)
graph.add_edge("handle_standard", END)

# Compile the graph
runnable = graph.compile()

print("✓ LangGraph workflow created and compiled!")
print("\n  Workflow structure:")
print("  START → categorize_request")
print("                ↓")
print("         [route_by_priority]")
print("           /          \\")
print("    handle_urgent  handle_standard")
print("           \\          /")
print("              END")

## Cell 8: Visualize the Graph

In [ ]:
from IPython.display import Image, display

print("=" * 60)
print("LANGGRAPH WORKFLOW VISUALIZATION")
print("=" * 60)

# Display the graph as PNG
try:
    display(Image(runnable.get_graph().draw_mermaid_png()))
    print("\n✓ Graph visualization displayed successfully!")
except Exception as e:
    print(f"Could not render PNG: {e}")
    print("\nMermaid syntax:")
    print(runnable.get_graph().draw_mermaid())

## Cell 9: Print Mermaid Diagram Syntax

In [ ]:
print("=" * 60)
print("MERMAID DIAGRAM SYNTAX")
print("=" * 60)
print("\nCopy this to any Mermaid-compatible viewer:\n")
print(runnable.get_graph().draw_mermaid())

## Cell 10: Test Case 1 - Urgent Request (Priority 1)

Simulating a high-priority security issue.

In [ ]:
print("=" * 60)
print("TEST CASE 1: URGENT REQUEST (Priority 1)")
print("=" * 60)

urgent_request = {
    "message": "My account was hacked! Urgent help needed.",
    "priority": 1,
    "status": "new"
}

print(f"\nInput: {urgent_request}\n")
print("-" * 40)

result1 = runnable.invoke(urgent_request)

print("-" * 40)
print(f"\n✓ Final Result: {result1}")

## Cell 11: Test Case 2 - Standard Request (Priority 3)

Simulating a low-priority password reset request.

In [ ]:
print("=" * 60)
print("TEST CASE 2: STANDARD REQUEST (Priority 3)")
print("=" * 60)

standard_request = {
    "message": "I need help with password reset.",
    "priority": 3,
    "status": "new"
}

print(f"\nInput: {standard_request}\n")
print("-" * 40)

result2 = runnable.invoke(standard_request)

print("-" * 40)
print(f"\n✓ Final Result: {result2}")

## Cell 12: Test Case 3 - Medium Priority Request (Priority 2)

Testing the boundary condition - Priority 2 should still go to urgent.

In [ ]:
print("=" * 60)
print("TEST CASE 3: MEDIUM PRIORITY REQUEST (Priority 2)")
print("=" * 60)

medium_request = {
    "message": "Payment failed and I have an important deadline!",
    "priority": 2,
    "status": "new"
}

print(f"\nInput: {medium_request}\n")
print("-" * 40)

result3 = runnable.invoke(medium_request)

print("-" * 40)
print(f"\n✓ Final Result: {result3}")

## Cell 13: Interactive Test - Try Your Own Request

In [ ]:
# Modify these values to test different scenarios
your_message = "I have a billing question about my subscription."
your_priority = 3  # Change to 1, 2, or 3

print("=" * 60)
print("INTERACTIVE TEST: YOUR CUSTOM REQUEST")
print("=" * 60)

custom_request = {
    "message": your_message,
    "priority": your_priority,
    "status": "new"
}

print(f"\nInput: {custom_request}\n")
print("-" * 40)

result_custom = runnable.invoke(custom_request)

print("-" * 40)
print(f"\n✓ Final Result: {result_custom}")

## Cell 14: Graph Structure Details

In [ ]:
print("=" * 60)
print("GRAPH STRUCTURE DETAILS")
print("=" * 60)

graph_obj = runnable.get_graph()

print("\n📍 NODES:")
for node in graph_obj.nodes:
    print(f"  • {node}")

print("\n🔗 EDGES:")
for edge in graph_obj.edges:
    print(f"  • {edge}")

## Summary

This notebook demonstrated:

1. **State Definition**: Using `TypedDict` to define the structure of data flowing through the graph

2. **Conditional Routing**: Using `add_conditional_edges()` to route requests based on priority level

3. **Graph Visualization**: Multiple ways to visualize the workflow (PNG, Mermaid, ASCII)

4. **Testing**: Different test cases showing how requests are routed based on priority

### Key LangGraph Concepts:
- `StateGraph`: The main graph class
- `add_node()`: Add processing nodes
- `add_edge()`: Add fixed edges between nodes
- `add_conditional_edges()`: Add dynamic routing based on state
- `START` and `END`: Special nodes for entry and exit points